In [14]:
import cv2
import numpy as np
import plotly.graph_objects as go

In [15]:
def load_images(rgb_path, depth_path):
    rgb = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
    depth = cv2.imread(depth_path, cv2.IMREAD_GRAYSCALE)

    if rgb is None:
        raise ValueError(f"Could not read RGB image: {rgb_path}")
    if depth is None:
        raise ValueError(f"Could not read depth image: {depth_path}")

    if rgb.shape[:2] != depth.shape[:2]:
        raise ValueError(f"Size mismatch. RGB: {rgb.shape[:2]}, Depth: {depth.shape[:2]}")

    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    return rgb, depth


def estimate_intrinsics(width, height, fov_deg=90.0):
    cx = width / 2.0
    cy = height / 2.0
    fov_rad = np.deg2rad(fov_deg)
    fx = width / (2.0 * np.tan(fov_rad / 2.0))
    fy = fx

    K = np.array([
        [fx, 0.0, cx],
        [0.0, fy, cy],
        [0.0, 0.0, 1.0]
    ], dtype=np.float32)

    return K


def depth_gray_to_distance(depth_gray, near_dist=1.0, far_dist=50.0):
    d = depth_gray.astype(np.float32) / 255.0
    d_inv = 1.0 - d
    Z = near_dist + d_inv * (far_dist - near_dist)
    return Z


def create_valid_mask(rgb, depth_gray, min_depth_gray=5, upper_remove_ratio=0.35):
    h, w = depth_gray.shape
    valid = np.ones((h, w), dtype=np.uint8)

    valid[depth_gray <= min_depth_gray] = 0

    upper_limit = int(upper_remove_ratio * h)
    valid[:upper_limit, :][depth_gray[:upper_limit, :] < 20] = 0

    r = rgb[:, :, 0].astype(np.float32)
    g = rgb[:, :, 1].astype(np.float32)
    b = rgb[:, :, 2].astype(np.float32)

    blue_sky = (b > 100) & (b > g + 10) & (b > r + 10)
    valid[:upper_limit, :][blue_sky[:upper_limit, :]] = 0

    return valid


def backproject_to_point_cloud(rgb, Z, K, valid_mask, stride=2):
    h, w = Z.shape
    fx = K[0, 0]
    fy = K[1, 1]
    cx = K[0, 2]
    cy = K[1, 2]

    points = []
    colors = []

    for v in range(0, h, stride):
        for u in range(0, w, stride):
            if valid_mask[v, u] == 0:
                continue

            z = Z[v, u]
            if z <= 0:
                continue

            x = (u - cx) * z / fx
            y = (v - cy) * z / fy

            points.append([x, y, z])
            colors.append(rgb[v, u])

    return np.array(points, dtype=np.float32), np.array(colors, dtype=np.uint8)

In [16]:
rgb_path = "front_U1wVHDYpSE7wIUQeItvlhg,41.852282,-87.646483,_input.png"
depth_path = "front_U1wVHDYpSE7wIUQeItvlhg,41.852282,-87.646483,_raw.png"

rgb, depth = load_images(rgb_path, depth_path)
h, w = depth.shape

K = estimate_intrinsics(w, h, fov_deg=90.0)
Z = depth_gray_to_distance(depth, near_dist=1.0, far_dist=50.0)
valid_mask = create_valid_mask(rgb, depth, min_depth_gray=5, upper_remove_ratio=0.35)

points, colors = backproject_to_point_cloud(
    rgb=rgb,
    Z=Z,
    K=K,
    valid_mask=valid_mask,
    stride=2
)

print("Point cloud shape:", points.shape)

Point cloud shape: (51926, 3)


In [17]:
def rgb_to_hex(colors):
    return [f"rgb({r},{g},{b})" for r, g, b in colors]


def show_interactive_point_cloud(points, colors, max_points=30000, point_size=2):
    n = len(points)

    if n > max_points:
        idx = np.random.choice(n, max_points, replace=False)
        pts = points[idx]
        cols = colors[idx]
    else:
        pts = points
        cols = colors

    plot_colors = rgb_to_hex(cols)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 2],
                z=-pts[:, 1],
                mode="markers",
                marker=dict(
                    size=point_size,
                    color=plot_colors,
                    opacity=0.8
                )
            )
        ]
    )

    fig.update_layout(
        title="Interactive 3D Point Cloud",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Z",
            zaxis_title="Height",
            aspectmode="data"
        ),
        width=1000,
        height=800
    )

    fig.show()

In [21]:
show_interactive_point_cloud(points, colors, max_points=30000, point_size=2)